In [ ]:
# IMPORTANT: Si vous modifiez agents/base_agents.py, redémarrez le kernel (Kernel -> Restart Kernel)
# pour que les changements soient pris en compte.

import os
import torch
import json
import traceback
import importlib
import sys

# Recharger le module agents.base_agents pour s'assurer d'utiliser la dernière version
if 'agents.base_agents' in sys.modules:
    importlib.reload(sys.modules['agents.base_agents'])

from huggingface_hub import login
from agents.base_agents import OrchestratorAgent, ResearcherAgent, CodeWriterAgent, CriticAgent, BaseAgent 

# ==============================================================================
# CONFIGURATION DE L'ACCÈS HUGGING FACE
# ==============================================================================
# Remplace le token codé en dur par la lecture depuis les variables d'environnement.
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Modèle utilisé pour l'entraînement

if not HF_TOKEN:
    print("⚠️ HF_TOKEN non défini. Si vous testez des modèles gated, exportez HF_TOKEN dans votre environnement.")
    print("   Exemple (Linux/macOS): export HF_TOKEN='hf_...'")
    print("   Exemple (Windows PowerShell): $env:HF_TOKEN='hf_...'")
    print(f"   Note: Le modèle {MODEL_NAME} n'est normalement pas gated, mais le token peut être utile.")
else:
    if not HF_TOKEN.startswith("hf_"):
        print("⚠️ Le format du token semble invalide.")
    else:
        try:
            login(token=HF_TOKEN, add_to_git_credential=True)
            print("✅ Authentification Hugging Face réussie.")
        except Exception as e:
            print(f"❌ Échec de l'authentification : {e}")
            print("   Si le modèle est gated, assurez-vous d'avoir demandé/accepté l'accès sur Hugging Face.")

# ==============================================================================
# CONFIGURATION MATÉRIELLE
# ==============================================================================
if torch.cuda.is_available():
    print("🚀 CUDA (GPU) disponible. Chargement sur GPU.")
    device = "cuda"
else:
    print("🐢 Aucun GPU trouvé. Utilisation du CPU (Lent).")
    device = "cpu"

# ==============================================================================
# CONFIGURATION DE MODE RAPIDE
# ==============================================================================
# Définir FAST_MODE=True pour accélérer les tests (moins de tokens, pas de retry)
FAST_MODE = os.environ.get("FAST_MODE", "False").lower() == "true"

# ==============================================================================
# FONCTION DE VALIDATION DES RÉSULTATS
# ==============================================================================

def validate_test_result(agent_name: str, result: dict) -> dict:
    """
    Valide le résultat d'un test et retourne un rapport détaillé.
    
    Args:
        agent_name: Nom de l'agent testé
        result: Résultat JSON de l'agent
    
    Returns:
        Dict avec 'success': bool, 'message': str, 'details': dict
    """
    validation = {
        'success': False,
        'message': '',
        'details': {},
        'score': 0,
        'max_score': 0
    }
    
    # Vérifier si c'est une erreur
    if result.get("action_type") == "ERROR":
        validation['message'] = f"❌ ÉCHEC : {result.get('error_message', 'Erreur inconnue')}"
        return validation
    
    # Définir les critères par agent
    if agent_name.lower() == "orchestrator":
        required_keys = ["delegated_agent", "instruction", "final_answer"]
        validation['max_score'] = 3
        
        # Vérifier les clés
        for key in required_keys:
            if key in result:
                validation['score'] += 1
                validation['details'][key] = "✅ Présent"
                
                # Validation spécifique pour delegated_agent
                if key == "delegated_agent":
                    val = result[key]
                    valid_values = ["Researcher", "CodeWriter", "Critic", "FINISHED"]
                    if val in valid_values:
                        validation['details'][f"{key}_value"] = f"✅ Valeur valide: {val}"
                    elif val:
                        validation['details'][f"{key}_value"] = f"⚠️ Valeur: {val} (normalisation possible)"
                    else:
                        validation['details'][f"{key}_value"] = "❌ Vide ou None"
            else:
                validation['details'][key] = "❌ Manquant"
        
        validation['success'] = validation['score'] == validation['max_score']
        if validation['success']:
            validation['message'] = f"✅ SUCCÈS : Toutes les clés requises présentes ({validation['score']}/{validation['max_score']})"
        else:
            validation['message'] = f"⚠️ PARTIEL : {validation['score']}/{validation['max_score']} clés présentes"
    
    elif agent_name.lower() == "researcher":
        required_keys = ["research_query", "final_answer"]
        validation['max_score'] = 2
        
        for key in required_keys:
            if key in result:
                validation['score'] += 1
                val = result[key]
                if val and val.strip():
                    validation['details'][key] = f"✅ Présent avec valeur: {val[:50]}..."
                else:
                    validation['details'][key] = "⚠️ Présent mais vide"
            else:
                validation['details'][key] = "❌ Manquant"
        
        validation['success'] = validation['score'] == validation['max_score']
        if validation['success']:
            validation['message'] = f"✅ SUCCÈS : Toutes les clés requises présentes ({validation['score']}/{validation['max_score']})"
        else:
            validation['message'] = f"⚠️ PARTIEL : {validation['score']}/{validation['max_score']} clés présentes"
    
    elif agent_name.lower() in ["codewriter", "code_writer"]:
        required_keys = ["python_code", "result_explanation"]
        validation['max_score'] = 2
        
        for key in required_keys:
            if key in result:
                validation['score'] += 1
                val = result[key]
                if key == "python_code" and val and val.strip():
                    validation['details'][key] = f"✅ Présent avec code ({len(val)} caractères)"
                elif key == "result_explanation":
                    if val:
                        validation['details'][key] = f"✅ Présent avec explication"
                    else:
                        validation['details'][key] = "⚠️ Présent mais vide (acceptable)"
                else:
                    validation['details'][key] = "❌ Présent mais vide"
            else:
                validation['details'][key] = "❌ Manquant"
        
        validation['success'] = validation['score'] == validation['max_score'] and result.get("python_code", "").strip()
        if validation['success']:
            validation['message'] = f"✅ SUCCÈS : Code Python valide ({validation['score']}/{validation['max_score']})"
        else:
            validation['message'] = f"⚠️ PARTIEL : {validation['score']}/{validation['max_score']} clés présentes"
    
    elif agent_name.lower() == "critic":
        required_keys = ["critique_ok", "suggestions"]
        validation['max_score'] = 2
        
        for key in required_keys:
            if key in result:
                validation['score'] += 1
                val = result[key]
                if key == "critique_ok":
                    if isinstance(val, bool):
                        validation['details'][key] = f"✅ Boolean correct: {val}"
                    else:
                        validation['details'][key] = f"⚠️ Type incorrect: {type(val).__name__} (sera normalisé)"
                elif key == "suggestions":
                    if val:
                        validation['details'][key] = f"✅ Présent avec suggestions"
                    else:
                        validation['details'][key] = "⚠️ Présent mais vide (acceptable)"
            else:
                validation['details'][key] = "❌ Manquant"
        
        validation['success'] = (
            validation['score'] == validation['max_score'] and 
            isinstance(result.get("critique_ok"), bool)
        )
        if validation['success']:
            validation['message'] = f"✅ SUCCÈS : Critique valide ({validation['score']}/{validation['max_score']})"
        else:
            validation['message'] = f"⚠️ PARTIEL : {validation['score']}/{validation['max_score']} clés présentes"
    
    else:
        validation['message'] = f"⚠️ Agent '{agent_name}' non reconnu pour validation"
    
    return validation

def test_agent(agent_class, test_query: str, max_retries: int = 1, fast_mode: bool = None):
    """
    Initialise un agent et teste sa capacité à répondre en JSON.
    Détecte spécifiquement les erreurs liées aux modèles gated/403 et propose des actions.
    
    Args:
        agent_class: Classe de l'agent à tester
        test_query: Requête de test
        max_retries: Nombre de tentatives supplémentaires en cas d'échec
        fast_mode: Si True, utilise le mode rapide (None = utilise FAST_MODE global)
    """
    if fast_mode is None:
        fast_mode = FAST_MODE
    
    if fast_mode:
        max_retries = 0  # Pas de retry en mode rapide
        print("⚡ Mode rapide activé (moins de tokens, pas de retry)")
    agent_name = agent_class.__name__.replace("Agent", "")
    print(f"\n" + "="*60)
    print(f"🧪 Test de l'agent : {agent_name}")
    print("-" * 60)
    
    try:
        # L'initialisation peut lever une erreur si le modèle est gated / accès refusé
        agent_instance = agent_class()
    except Exception as e:
        err = str(e).lower()
        # Détection basique d'erreurs d'accès à repo gated / 403
        if any(keyword in err for keyword in ("gated", "403", "access to model", "not in the authorized list", "cannot access gated", "gated repo")):
            print(f"\n❌ ACCÈS RESTREINT ({agent_name}) : Le chargement a échoué à cause d'un modèle gated.")
            print("   Actions possibles :")
            print(f"    - Vérifiez l'accès au modèle sur Hugging Face: https://huggingface.co/{MODEL_NAME}")
            print("    - Exportez HF_TOKEN (token ayant accès) dans votre environnement et relancez.")
            print("    - Définissez SKIP_GATED_MODELS=1 pour ignorer automatiquement les agents dépendants de modèles restreints.")
            if os.environ.get("SKIP_GATED_MODELS") == "1":
                print("   -> SKIP_GATED_MODELS=1 détecté : cet agent sera ignoré.")
                return
            else:
                print("   -> Cet agent est sauté pour éviter une erreur bloquante.")
                return
        else:
            # Erreur non liée à l'accès restreint : la laisser remonter comme échec critique mais avec message utile.
            print(f"\n❌ ÉCHEC CRITIQUE ({agent_name}) lors de l'initialisation :")
            print(f"   Erreur: {e}")
            print("   Traceback complet:")
            import traceback as tb
            tb.print_exc()
            print("\n   Notes:")
            print(f"   - Vérifiez que les checkpoints LoRA existent dans checkpoints/{agent_name.lower().replace('codewriter', 'code_writer')}_lora/")
            print(f"   - Vérifiez que le modèle {MODEL_NAME} est accessible")
            print("   - Vérifiez les logs complets et que les dépendances sont installées")
            return

    # Tentatives multiples pour améliorer les chances de succès
    for attempt in range(max_retries + 1):
        try:
            if attempt > 0:
                print(f"\n🔄 Tentative {attempt + 1}/{max_retries + 1}...")
            
            print(f"\n📝 Requête : {test_query}")
            print("⏳ Génération de la réponse...")
            
            # Appel de la méthode act() avec mode rapide
            result_action = agent_instance.act(test_query, fast_mode=fast_mode)
            
            print("-" * 60)
            print(f"🔥 Résultat JSON :")
            print(json.dumps(result_action, indent=4, ensure_ascii=False))
            
            if result_action.get("action_type") == "ERROR":
                error_msg = result_action.get('error_message', 'Unknown error')
                print(f"\n❌ ÉCHEC DU PARSING : {error_msg}")
                
                if "raw_output" in result_action:
                    raw_output = result_action["raw_output"]
                    print("\n📄 Sortie brute du modèle:")
                    print(raw_output[:800] + "..." if len(raw_output) > 800 else raw_output)
                    
                    # Analyse de la sortie brute
                    print("\n🔍 Analyse de la sortie:")
                    if raw_output.strip().startswith('{'):
                        print("   ✅ La sortie commence par '{' (bon signe)")
                    else:
                        print("   ⚠️ La sortie ne commence pas par '{'")
                        first_brace = raw_output.find('{')
                        if first_brace >= 0:
                            print(f"   → Premier '{{' trouvé à la position {first_brace}")
                            print(f"   → Texte avant: '{raw_output[:first_brace][:50]}...'")
                    
                    # Vérifier la présence de clés JSON attendues
                    if agent_name == "Orchestrator":
                        expected_keys = ["delegated_agent", "instruction", "final_answer"]
                    elif agent_name == "Researcher":
                        expected_keys = ["research_query", "final_answer"]
                    elif agent_name == "CodeWriter":
                        expected_keys = ["python_code", "result_explanation"]
                    elif agent_name == "Critic":
                        expected_keys = ["critique_ok", "suggestions"]
                    else:
                        expected_keys = []
                    
                    found_keys = []
                    for key in expected_keys:
                        if f'"{key}"' in raw_output or f"'{key}'" in raw_output:
                            found_keys.append(key)
                    
                    if found_keys:
                        print(f"   → Clés JSON trouvées dans la sortie: {found_keys}")
                    if len(found_keys) < len(expected_keys):
                        missing = set(expected_keys) - set(found_keys)
                        print(f"   ⚠️ Clés manquantes: {missing}")
                
                # Si ce n'est pas la dernière tentative, continuer
                if attempt < max_retries:
                    continue
                else:
                    print(f"\n⚠️ Toutes les tentatives ont échoué pour {agent_name}")
                    return
            else:
                print(f"\n✅ SUCCÈS : Format JSON valide pour {agent_name}.")
                # Validation spécifique selon l'agent
                if agent_name == "Orchestrator":
                    if "delegated_agent" in result_action:
                        print(f"   → Agent délégué: {result_action['delegated_agent']}")
                    if "instruction" in result_action:
                        print(f"   → Instruction: {result_action['instruction'][:80]}...")
                elif agent_name == "Researcher":
                    if "research_query" in result_action:
                        print(f"   → Requête de recherche: {result_action['research_query']}")
                    if "final_answer" in result_action and result_action['final_answer']:
                        print(f"   → Réponse finale: {result_action['final_answer'][:80]}...")
                elif agent_name == "CodeWriter":
                    if "python_code" in result_action:
                        code_len = len(result_action['python_code'])
                        print(f"   → Code Python généré ({code_len} caractères)")
                        print(f"   → Code: {result_action['python_code'][:100]}...")
                elif agent_name == "Critic":
                    if "critique_ok" in result_action:
                        print(f"   → Critique OK: {result_action['critique_ok']}")
                    if "suggestions" in result_action:
                        print(f"   → Suggestions: {result_action['suggestions'][:80]}...")
                return  # Succès, sortir de la boucle
                
        except Exception as e:
            print(f"\n❌ ÉCHEC CRITIQUE ({agent_name}) durant act() (tentative {attempt + 1}):")
            print(f"   Erreur: {e}")
            if attempt == max_retries:
                print("   Traceback complet:")
                import traceback as tb
                tb.print_exc()
                print(f"\n   Note: Vérifiez que le modèle {MODEL_NAME} est correctement chargé et que les checkpoints LoRA sont valides.")
                return

# ==============================================================================
# INTERFACE DE TEST INTERACTIVE
# ==============================================================================

def test_single_agent(agent_name: str, query: str, fast_mode: bool = None):
    """
    Teste un agent spécifique par son nom.
    
    Args:
        agent_name: Nom de l'agent ('orchestrator', 'researcher', 'code_writer', 'critic')
        query: Requête à tester
        fast_mode: Si True, utilise le mode rapide (None = utilise FAST_MODE global)
    """
    agent_map = {
        'orchestrator': OrchestratorAgent,
        'researcher': ResearcherAgent,
        'code_writer': CodeWriterAgent,
        'critic': CriticAgent
    }
    
    agent_name_lower = agent_name.lower().replace(' ', '_')
    if agent_name_lower not in agent_map:
        print(f"❌ Agent '{agent_name}' non trouvé. Agents disponibles: {list(agent_map.keys())}")
        return None
    
    agent_class = agent_map[agent_name_lower]
    test_agent(agent_class, query, max_retries=0 if (fast_mode if fast_mode is not None else FAST_MODE) else 1, fast_mode=fast_mode)

def test_workflow(query: str, max_iterations: int = 5):
    """
    Teste un workflow complet avec l'Orchestrator qui délègue aux autres agents.
    
    Args:
        query: Requête initiale pour l'Orchestrator
        max_iterations: Nombre maximum d'itérations du workflow
    """
    print("\n" + "="*70)
    print("🔄 TEST DE WORKFLOW MULTI-AGENT")
    print("="*70)
    print(f"Requête initiale: {query}\n")
    
    try:
        orchestrator = OrchestratorAgent()
        researcher = ResearcherAgent()
        code_writer = CodeWriterAgent()
        critic = CriticAgent()
        
        agents = {
            'Researcher': researcher,
            'CodeWriter': code_writer,
            'Critic': critic,
            'FINISHED': None
        }
        
        current_observation = query
        iteration = 0
        
        while iteration < max_iterations:
            iteration += 1
            print(f"\n{'='*70}")
            print(f"🔄 Itération {iteration}/{max_iterations}")
            print(f"{'='*70}")
            print(f"📝 Observation: {current_observation[:100]}...\n")
            
            # Action de l'Orchestrator
            orchestrator_action = orchestrator.act(current_observation)
            print(f"🎯 Orchestrator décide:")
            print(json.dumps(orchestrator_action, indent=2, ensure_ascii=False))
            
            if orchestrator_action.get("action_type") == "ERROR":
                print(f"\n❌ Erreur de l'Orchestrator: {orchestrator_action.get('error_message')}")
                break
            
            delegated_agent = orchestrator_action.get("delegated_agent", "").strip()
            instruction = orchestrator_action.get("instruction", "")
            final_answer = orchestrator_action.get("final_answer", "")
            
            if delegated_agent == "FINISHED" or final_answer:
                print(f"\n✅ Workflow terminé!")
                print(f"📄 Réponse finale: {final_answer}")
                break
            
            if delegated_agent not in agents:
                print(f"\n❌ Agent '{delegated_agent}' non reconnu")
                break
            
            if agents[delegated_agent] is None:
                print(f"\n✅ Workflow terminé (FINISHED)")
                break
            
            # Action de l'agent délégué
            print(f"\n🤖 {delegated_agent} exécute: {instruction[:80]}...")
            agent_action = agents[delegated_agent].act(instruction)
            print(f"📋 Résultat {delegated_agent}:")
            print(json.dumps(agent_action, indent=2, ensure_ascii=False)[:500])
            
            # Préparer la prochaine observation
            current_observation = f"Résultat de {delegated_agent}: {json.dumps(agent_action, ensure_ascii=False)}"
            
        if iteration >= max_iterations:
            print(f"\n⚠️ Nombre maximum d'itérations ({max_iterations}) atteint")
            
    except Exception as e:
        print(f"\n❌ Erreur lors du workflow: {e}")
        traceback.print_exc()

# ==============================================================================
# EXEMPLES DE TESTS PRÉDÉFINIS
# ==============================================================================

TEST_EXAMPLES = {
    "simple_orchestrator": {
        "agent": "orchestrator",
        "query": "Planifie une analyse comparative entre le Pixel 8 et l'iPhone 15."
    },
    "simple_researcher": {
        "agent": "researcher", 
        "query": "Cherche la date de sortie exacte du Google Pixel 8 Pro."
    },
    "simple_code_writer": {
        "agent": "code_writer",
        "query": "Fais un script Python pour calculer une remise de 15% sur un prix de 899€."
    },
    "simple_critic": {
        "agent": "critic",
        "query": "Évalue ceci : 'Le smartphone est cher mais puissant'."
    },
    "workflow_complex": {
        "workflow": True,
        "query": "Compare les caractéristiques techniques du Pixel 8 et de l'iPhone 15, puis calcule lequel est le meilleur rapport qualité-prix."
    }
}

# ==============================================================================
# FONCTION DE TEST RAPIDE
# ==============================================================================

def quick_test(example_name: str = None):
    """
    Test rapide avec des exemples prédéfinis ou interactif.
    
    Usage:
        quick_test("simple_orchestrator")  # Test un exemple
        quick_test()  # Menu interactif
    """
    if example_name and example_name in TEST_EXAMPLES:
        example = TEST_EXAMPLES[example_name]
        if "workflow" in example:
            test_workflow(example["query"])
        else:
            test_single_agent(example["agent"], example["query"])
    else:
        print("\n" + "="*60)
        print("🧪 MENU DE TEST INTERACTIF")
        print("="*60)
        print("\nExemples disponibles:")
        for i, (name, example) in enumerate(TEST_EXAMPLES.items(), 1):
            if "workflow" in example:
                print(f"  {i}. {name}: Workflow complet - {example['query'][:60]}...")
            else:
                print(f"  {i}. {name}: {example['agent']} - {example['query'][:60]}...")
        print(f"  {len(TEST_EXAMPLES)+1}. Tous les agents (test séquentiel)")
        print(f"  {len(TEST_EXAMPLES)+2}. Workflow complet (exemple)")
        print("\nUtilisez: quick_test('nom_exemple') pour tester un exemple spécifique")
        print("Ou utilisez test_single_agent('agent_name', 'query') pour un test personnalisé")

# ==============================================================================
# FONCTION DE TEST INTERACTIF SIMPLE
# ==============================================================================

def test_interactif(phrase: str, agent: str = "orchestrator", fast_mode: bool = None):
    """
    Fonction simple pour tester un agent avec votre propre phrase.
    
    Args:
        phrase: Votre phrase/requête
        agent: Nom de l'agent ('orchestrator', 'researcher', 'code_writer', 'critic')
        fast_mode: Si True, mode rapide (None = utilise FAST_MODE global)
    
    Exemples:
        test_interactif("Compare le Pixel 8 et l'iPhone 15")
        test_interactif("Cherche la date de sortie du Pixel 8", agent="researcher")
        test_interactif("Calcule 15% de 899", agent="code_writer")
    """
    print("\n" + "="*70)
    print(f"💬 TEST INTERACTIF - Agent: {agent.upper()}")
    print("="*70)
    print(f"📝 Votre phrase: {phrase}")
    print("="*70 + "\n")
    
    test_single_agent(agent, phrase, fast_mode=fast_mode)
    
    print("\n" + "="*70)
    print("✅ Test terminé")
    print("="*70)

def test_tous_agents(phrase: str, fast_mode: bool = None):
    """
    Teste votre phrase avec tous les agents.
    
    Args:
        phrase: Votre phrase/requête
        fast_mode: Si True, mode rapide
    """
    agents = ["orchestrator", "researcher", "code_writer", "critic"]
    
    print("\n" + "="*70)
    print(f"🔄 TEST AVEC TOUS LES AGENTS")
    print("="*70)
    print(f"📝 Votre phrase: {phrase}")
    print("="*70 + "\n")
    
    for agent in agents:
        print(f"\n{'='*70}")
        print(f"🤖 Test avec {agent.upper()}")
        print(f"{'='*70}")
        test_single_agent(agent, phrase, fast_mode=fast_mode)
        print("\n")
    
    print("="*70)
    print("✅ Tous les tests terminés")
    print("="*70)

# ==============================================================================
# EXÉCUTION PAR DÉFAUT
# ==============================================================================

if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 DÉMARRAGE DES TESTS DES AGENTS")
    print("="*60)
    print(f"Modèle de base: {MODEL_NAME}")
    print(f"Device: {device}")
    print("\n💡 Utilisez les fonctions suivantes pour tester:")
    print("   - quick_test('simple_orchestrator')  # Test un exemple")
    print("   - test_single_agent('orchestrator', 'votre requête')  # Test personnalisé")
    print("   - test_workflow('votre requête complexe')  # Workflow complet")
    print("\n⚡ Mode rapide:")
    print("   - Définir FAST_MODE=True dans l'environnement pour accélérer")
    print("   - Ou utiliser test_single_agent(..., fast_mode=True)")
    print(f"\n⚡ Mode rapide actuel: {FAST_MODE}")
    print("\n" + "-"*60)
    print("Exécution des tests par défaut...")
    print("-"*60 + "\n")
    
    # Tests par défaut (vous pouvez commenter ceux que vous ne voulez pas)
    default_tests = [
        ("orchestrator", "Planifie une analyse comparative entre le Pixel 8 et l'iPhone 15."),
        ("researcher", "Cherche la date de sortie exacte du Google Pixel 8 Pro."),
        ("code_writer", "Fais un script Python pour calculer une remise de 15% sur un prix de 899€."),
        ("critic", "Évalue ceci : 'Le smartphone est cher mais puissant'.")
    ]
    
    for agent_name, query in default_tests:
        test_single_agent(agent_name, query)
        print("\n" + "-"*60 + "\n")
    
    print("✅ TESTS TERMINÉS")
    print("="*60)

⚠️ HF_TOKEN non défini. Si vous testez des modèles gated, exportez HF_TOKEN dans votre environnement.
   Exemple (Linux/macOS): export HF_TOKEN='hf_...'
   Exemple (Windows PowerShell): $env:HF_TOKEN='hf_...'
   Note: Le modèle TinyLlama/TinyLlama-1.1B-Chat-v1.0 n'est normalement pas gated, mais le token peut être utile.
🐢 Aucun GPU trouvé. Utilisation du CPU (Lent).

🚀 DÉMARRAGE DES TESTS DES AGENTS
Modèle de base: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Device: cpu

💡 Utilisez les fonctions suivantes pour tester:
   - quick_test('simple_orchestrator')  # Test un exemple
   - test_single_agent('orchestrator', 'votre requête')  # Test personnalisé
   - test_workflow('votre requête complexe')  # Workflow complet

⚡ Mode rapide:
   - Définir FAST_MODE=True dans l'environnement pour accélérer
   - Ou utiliser test_single_agent(..., fast_mode=True)

⚡ Mode rapide actuel: False

------------------------------------------------------------
Exécution des tests par défaut...
-----------------------

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


------------------------------------------------------------
🔥 Résultat JSON :
{
    "suggestions": "Concretement des recommandations pour amÃ©liorer ou confirmer la suggestion.",
    "action_type": "ERROR",
    "error_message": "Critic action missing required keys (critique_ok, suggestions)."
}

❌ ÉCHEC DU PARSING : Critic action missing required keys (critique_ok, suggestions).

🔄 Tentative 2/2...

📝 Requête : Évalue ceci : 'Le smartphone est cher mais puissant'.
⏳ Génération de la réponse...
------------------------------------------------------------
🔥 Résultat JSON :
{
    "suggestions": "Concretement des recommandations pour amÃ©liorer ou confirmer la suggestion.",
    "action_type": "ERROR",
    "error_message": "Critic action missing required keys (critique_ok, suggestions)."
}

❌ ÉCHEC DU PARSING : Critic action missing required keys (critique_ok, suggestions).

⚠️ Toutes les tentatives ont échoué pour Critic

------------------------------------------------------------

✅ TES

# Guide d'utilisation des tests

## 🚀 Fonction Simple pour Tester avec Votre Phrase

### Test Interactif (Recommandé)
```python
# Testez avec votre propre phrase
test_interactif("Compare le Pixel 8 et l'iPhone 15")
test_interactif("Cherche la date de sortie du Pixel 8", agent="researcher")
test_interactif("Calcule 15% de 899", agent="code_writer")
test_interactif("Évalue ce code: def test(): return 1", agent="critic")
```

### Tester avec Tous les Agents
```python
# Teste votre phrase avec tous les agents
test_tous_agents("Compare le Pixel 8 et l'iPhone 15")
```

---

## Autres Fonctions Disponibles

### 1. Tester un agent individuel
```python
test_single_agent('orchestrator', 'Votre requête ici')
test_single_agent('researcher', 'Cherche des informations sur...')
test_single_agent('code_writer', 'Écris du code pour...')
test_single_agent('critic', 'Évalue cette solution: ...')
```

### 2. Tester un workflow complet
```python
test_workflow('Compare les smartphones Pixel 8 et iPhone 15')
```

### 3. Tests rapides avec exemples prédéfinis
```python
quick_test('simple_orchestrator')
quick_test('simple_researcher')
quick_test('workflow_complex')
```

### 4. Mode Rapide
```python
# Activer le mode rapide pour accélérer
test_interactif("Votre phrase", fast_mode=True)
```


In [ ]:
# ==============================================================================
# 🎯 EXEMPLE : TEST INTERACTIF AVEC VOTRE PHRASE
# ==============================================================================
# Modifiez la phrase ci-dessous pour tester avec votre propre texte

ma_phrase = "Compare les caractéristiques du Pixel 8 et de l'iPhone 15"

# Choisissez l'agent à tester :
# - "orchestrator" : Planifie et délègue les tâches
# - "researcher" : Recherche des informations
# - "code_writer" : Génère du code Python
# - "critic" : Évalue et critique

agent_choisi = "orchestrator"  # Changez ici l'agent

# Testez avec votre phrase
test_interactif(ma_phrase, agent=agent_choisi, fast_mode=True)


In [ ]:
# ==============================================================================
# 🎯 EXEMPLE : TESTER AVEC TOUS LES AGENTS
# ==============================================================================
# Teste votre phrase avec tous les agents en une fois

ma_phrase = "Compare les caractéristiques du Pixel 8 et de l'iPhone 15"

# Décommentez la ligne suivante pour tester avec tous les agents
# test_tous_agents(ma_phrase, fast_mode=True)


In [ ]:
# ==============================================================================
# 🎯 EXEMPLE : TESTER UN AGENT SPÉCIFIQUE
# ==============================================================================
# Exemples pour chaque type d'agent

# Pour Researcher (recherche d'informations)
# test_interactif("Quelle est la date de sortie du Google Pixel 8 Pro?", agent="researcher", fast_mode=True)

# Pour CodeWriter (génération de code)
# test_interactif("Écris un script Python pour calculer une remise de 15% sur 899€", agent="code_writer", fast_mode=True)

# Pour Critic (évaluation)
# test_interactif("Évalue ce code: def calculate(x): return x * 2", agent="critic", fast_mode=True)

# Pour Orchestrator (planification)
# test_interactif("Planifie une analyse comparative entre le Pixel 8 et l'iPhone 15", agent="orchestrator", fast_mode=True)


In [3]:
# Exemple 1: Tester un agent spécifique
# Décommentez la ligne souhaitée pour tester :

test_single_agent('orchestrator', 'Planifie une analyse comparative entre le Pixel 8 et l\'iPhone 15.')
test_single_agent('researcher', 'Cherche la date de sortie exacte du Google Pixel 8 Pro.')
test_single_agent('code_writer', 'Fais un script Python pour calculer une remise de 15% sur un prix de 899€.')
test_single_agent('critic', 'Évalue ceci : "Le smartphone est cher mais puissant".')



🧪 Test de l'agent : Orchestrator
------------------------------------------------------------

--- Chargement de l'agent Orchestrator ---
Modèle de base: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Device: cpu
Application des poids LoRA depuis : checkpoints/orchestrator_lora
   Chargement des poids depuis adapter_model.safetensors
   ✅ Poids LoRA chargés avec succès
✅ Modèle Orchestrator prêt.

📝 Requête : Planifie une analyse comparative entre le Pixel 8 et l'iPhone 15.
⏳ Génération de la réponse...
------------------------------------------------------------
🔥 Résultat JSON :
{
    "data": {
        "delegation_agency": null,
        "question": "[{\\\\*] Find information about \\(Pixel 8\" and iPhon 14\", what is it? ",
        "operation": "="
    },
    "operands": [
        null
    ],
    "operator": "==",
    "result": "true",
    "action_type": "ERROR",
    "error_message": "Orchestrator action missing required keys (delegated_agent, instruction, final_answer)."
}

❌ ÉCHEC DU PARSING 

KeyboardInterrupt: 

In [ ]:
# Exemple 2: Tester un workflow complet (multi-agent)
# L'Orchestrator délègue aux autres agents automatiquement

test_workflow('Compare les caractéristiques techniques du Pixel 8 et de l\'iPhone 15')


In [ ]:
# Exemple 3: Tests rapides avec exemples prédéfinis
# Voir tous les exemples disponibles :
quick_test()


In [ ]:
# Exemple 4: Tester un exemple spécifique
# Décommentez pour tester :

quick_test('simple_orchestrator')
quick_test('simple_researcher')
quick_test('simple_code_writer')
quick_test('simple_critic')
quick_test('workflow_complex')
